# ADT Contracts

```{contents}
:local:
:depth: 2
```

An **abstract data type** starts with behavior. Before deciding whether data should be stored in an array, linked nodes, or a hash table, we decide what operations the type promises to support.


```{index} Describe the Observable History First
```



```{index} Describe the Observable History First
```

## Describe the Observable History First



An undo history is not defined by an array or a linked node. It is defined by what a client can observe after a sequence of operations. If the client records A, records B, and then undoes once, it expects B to be removed before A. An implementation can store those entries in different ways while preserving that behavior. This separates the abstract data type, the behavioral contract, from the data structure used to realize it.

Write the history on paper before writing a class. For each operation, record the result, the resulting abstract contents, and the empty/failure policy. If the contract says an unsuccessful TryPop leaves state unchanged, returning false after already decrementing Count violates the contract even if later tests happen to restore the count.

| Operation | Result | Abstract history, oldest to newest |
| --- | --- | --- |
| Start | Empty | [] |
| Push A | Recorded | [A] |
| Push B | Recorded | [A, B] |
| TryPop | Success, B | [A] |
| TryPop | Success, A | [] |
| TryPop | Failure | [] |

The empty failure is part of the behavior, not an implementation detail to settle later. A different ADT might throw on an empty Pop. Both can be coherent contracts, but a client written for one cannot assume the other. Also specify whether null entries, duplicates, or a fixed capacity are allowed. These policies determine which tests should succeed and what failure must preserve.

For related explanation, see the further reading below.[^development-reading]


```{index} Separate Client Knowledge from Implementation Knowledge
```



```{index} Separate Client Knowledge from Implementation Knowledge
```

## Separate Client Knowledge from Implementation Knowledge



A C# interface can express operation names and types, for example a success-returning method with an out result. It cannot by itself express every ordering rule or state invariant. Two classes can compile against the same interface and still behave differently. Tests and explanations connect the syntax to the promised history.

The client should not need to know the backing array's capacity or the head node's field name. If a capacity change affects returned values or exposes unused slots, representation details have leaked through the abstraction. Conversely, the implementer must know enough about storage to maintain its invariants. Abstraction assigns different responsibilities to the two roles; it does not mean that implementation details are unimportant.

Guided exercise: keep the interface unchanged but change removal to return A before B. Identify the first row at which this behavior differs from the specified history. Then explain why a test that checks only Count would miss the defect. Extend the history with a failed removal followed by a new push; this checks whether failure left the structure reusable.

```{dropdown} Contract checkpoint
The first successful removal must return B. A queue-like implementation could keep the right counts while returning the wrong element. Checking returned values, order, state after failure, and later reuse tests more of the contract than checking size alone.
```


```{index} abstract data type
```
```{index} contract; data structure
```



```{index} What an ADT Specifies
```

## What an ADT Specifies



An ADT describes a set of values and the operations allowed on those values. It does **not** require one particular internal representation.

For a stack, the contract might say:

| Operation | Meaning |
| --- | --- |
| `Push(item)` | Add an item to the top. |
| `Pop()` | Remove and return the top item. |
| `Peek()` | Return the top item without removing it. |
| `Count` | Report how many items are stored. |

This contract says nothing about arrays or linked lists. That separation is the point: client code should depend on the behavior it needs, not on accidental implementation details.

The vocabulary behind that table is worth fixing now, because the rest of the book uses it precisely. A *type* is a set of values: the integers, the Booleans, the strings. A *data item* is one value drawn from a type, such as the account balance 1250.75. A *data type* is a type together with the operations allowed on it: integers with addition and comparison, strings with concatenation and length. An *abstract data type* is a data type described only by its operations and their observable results, with no commitment to how values are stored. A *data structure* is one concrete implementation of an ADT in a program. In an object-oriented language the ADT and its data structure usually arrive together as one class: the public members express the contract, and the private fields hold the representation.

| Term | What it names | Stack example |
| --- | --- | --- |
| Type | A set of values | All possible undo histories |
| Data item | One value from a type | The history [open file, rename variable] |
| Data type | A type plus its operations | Histories with Push, Pop, Peek, Count |
| Abstract data type | The operations and their promised results, without storage | The contract table above |
| Data structure | A concrete implementation of the ADT | `ListStack<T>` below, or an array with a top index |

The same separation applies to a lookup table. A *dictionary ADT* promises three operations: add a record under a key, find the record for a key, and remove a key. A product catalog, a customer directory, and a parts list all need exactly this contract. The data structures that can fulfill it are very different: a sorted array searched by halving, a hash table that computes a slot from the key, or a binary search tree that branches left or right at each node. All three return the same answer to `Find("P-100")`; they differ in how much work each operation costs and in what extra memory they spend. Chapters 17 and 19 implement two of them. Even the word *array* has both meanings: as a data type it is a numbered collection accessed by index, and as a data structure it is a contiguous block of memory. A program can present an array-like interface over storage that is not contiguous at all, which is how sparse spreadsheets and very large matrices are often built.

```{index} interface; ADT
```



```{index} Interfaces as Contracts
```

## Interfaces as Contracts



In C#, an `interface` is a natural way to write an ADT contract. It names the operations, their parameter types, and their result types, but it contains no fields and no method bodies, so it cannot commit to a storage strategy. A class then *implements* the interface by supplying a body for every member. The general shape is:

```csharp
interface IName<T>            // the contract: member signatures only
{
    int Count { get; }
    void Operation(T item);
    T Query();
}

class Implementation<T> : IName<T>   // one data structure that fulfills it
{
    // private fields hold the representation
    // public members give every interface member a body
}
```

The compiler enforces the syntactic half of the contract: a class that claims `: IStack<T>` but omits `Peek` does not compile. The behavioral half, that `Pop` returns the most recent `Push`, is not checked by the compiler; tests and the written contract carry it. The example below declares the stack contract, implements it with a `List<T>` that appends at the end and removes from the end, and drives it through a variable whose declared type is the interface.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
IStack<string> history = new ListStack<string>();
history.Push("open file");
history.Push("rename variable");
Console.WriteLine(history.Pop());
Console.WriteLine(history.Peek());

interface IStack<T>
{
    int Count { get; }
    void Push(T item);
    T Pop();
    T Peek();
}

class ListStack<T> : IStack<T>
{
    private readonly List<T> items = new();
    public int Count => items.Count;

    public void Push(T item) => items.Add(item);

    public T Pop()
    {
        if (items.Count == 0) throw new InvalidOperationException("Stack is empty.");
        int last = items.Count - 1;
        T value = items[last];
        items.RemoveAt(last);
        return value;
    }

    public T Peek()
    {
        if (items.Count == 0) throw new InvalidOperationException("Stack is empty.");
        return items[^1];
    }
}

rename variable
open file


The output is `rename variable` followed by `open file`. Two pushes recorded `open file` and then `rename variable`; `Pop` removed and returned the most recent entry, and `Peek` then reported the entry underneath without removing it, so `Count` is still 1 after the two lines print. A third `Pop` would throw `InvalidOperationException`, which is the empty-stack policy this contract chose.

Notice what the first line does not say. `history` is declared as `IStack<string>`, not `ListStack<string>`, so nothing after that line depends on the `List<T>` inside. The private `items` field is the representation; `Count`, `Push`, `Pop`, and `Peek` are the contract. Appending at the end of the list and removing from the end keeps both operations cheap, but that is a property of this data structure, not of the stack ADT. A common mistake is to let the representation leak: adding a public `Items` property for convenience would let a client reorder the history behind the stack's back and break the rule that `Pop` returns what `Peek` showed.

```{index} client code
```



```{index} Client Code Depends on Behavior
```

## Client Code Depends on Behavior



**Client code** is code that uses a type. If client code only needs stack behavior, it should not care whether the stack is backed by a `List<T>`, an array, or linked nodes.

That means this is a better variable type for many methods:

```csharp
static void UndoLastAction(IStack<string> actions)
```

than this:

```csharp
static void UndoLastAction(ListStack<string> actions)
```

The first version accepts any stack implementation that satisfies the contract.

Depending on the interface buys three things. First, the method accepts any current or future implementation, so switching from a list-backed stack to an array-backed one changes a single `new` expression at the call site. Second, tests can pass a tiny hand-written implementation that records every call, which is easier to check than a full data structure. Third, the method's own code becomes shorter to read, because the only operations it can call are the ones in the contract. The example below writes `CountUndoable` once and runs it against two different data structures. Only the `new` expressions differ.

In [ ]:
using System;
using System.Collections.Generic;

Console.WriteLine(CountUndoable(Fill(new ListStack<string>())));
Console.WriteLine(CountUndoable(Fill(new LinkedStack<string>())));

static IStack<string> Fill(IStack<string> history)
{
    history.Push("open file");
    history.Push("rename variable");
    history.Push("save");
    return history;
}

// Depends only on the contract: it works for every IStack<string>.
static int CountUndoable(IStack<string> history)
{
    int undone = 0;
    while (history.Count > 0)
    {
        history.Pop();
        undone++;
    }
    return undone;
}

interface IStack<T>
{
    int Count { get; }
    void Push(T item);
    T Pop();
    T Peek();
}

class ListStack<T> : IStack<T>
{
    private readonly List<T> items = new();
    public int Count => items.Count;
    public void Push(T item) => items.Add(item);
    public T Pop()
    {
        if (items.Count == 0) throw new InvalidOperationException("Stack is empty.");
        T value = items[^1];
        items.RemoveAt(items.Count - 1);
        return value;
    }
    public T Peek() => items.Count == 0 ? throw new InvalidOperationException("Stack is empty.") : items[^1];
}

class LinkedStack<T> : IStack<T>
{
    private sealed class Node { public T Value; public Node? Next; public Node(T v, Node? n) { Value = v; Next = n; } }
    private Node? top;
    public int Count { get; private set; }
    public void Push(T item) { top = new Node(item, top); Count++; }
    public T Pop()
    {
        if (top is null) throw new InvalidOperationException("Stack is empty.");
        T value = top.Value;
        top = top.Next;
        Count--;
        return value;
    }
    public T Peek() => top is null ? throw new InvalidOperationException("Stack is empty.") : top.Value;
}

3
3


Both lines print `3`. `CountUndoable` never learns which class it received: the list-backed stack removes from the end of a `List<T>`, the linked stack advances a `top` reference, and the method sees only `Count` and `Pop`. If `CountUndoable` had been declared with a `ListStack<string>` parameter, the second call would not compile even though the linked stack behaves identically. The variation to try: add a fourth `Push` inside `Fill` and confirm that both counts change together, which is exactly the property a shared contract is supposed to give you.

```{index} invariant
```



```{index} Contracts Include Rules
```

## Contracts Include Rules



A contract includes more than method names. It also includes rules:

- `Pop` removes the same item that `Peek` would return.
- `Count` increases after `Push` and decreases after `Pop`.
- `Pop` and `Peek` on an empty stack are errors.

A rule that must remain true for the object to be valid is called an **invariant**. Data-structure code often exists to preserve invariants while operations run quickly.

It helps to sort the rules of a contract into three kinds, because each is checked at a different moment.

| Kind of rule | When it must hold | Stack example | Who is responsible |
| --- | --- | --- | --- |
| Precondition | Before an operation is called | `Pop` requires `Count > 0` | The caller |
| Postcondition | After the operation returns | After `Push(x)`, `Peek()` returns `x` and `Count` grew by one | The implementation |
| Invariant | Between every pair of operations | `Count` equals the number of items that can still be popped | The implementation |

A precondition is a demand on the client; the implementation may check it and throw, or may state that the result is undefined when it is violated. A postcondition is the promise the client relies on. An invariant is the internal consistency that makes every postcondition achievable. When you implement a data structure, the useful habit is to write the invariant as a comment above the fields and ask of every method, "if this was true when I was called, is it still true when I return?" A method that temporarily breaks the invariant, for example by decrementing `Count` before confirming the stack was nonempty, must restore it on every path, including the failure path.

```{index} Code Tracing: Follow a Stack History
```

### Code Tracing: Follow a Stack History

Read the program below without running it. Write down the abstract history after each statement, then predict the four printed lines. The implementation is the `ListStack<T>` from earlier in this section; only the contract matters for the prediction.

```csharp
IStack<string> edits = new ListStack<string>();
edits.Push("insert row");
edits.Push("format cell");
edits.Push("delete column");
Console.WriteLine(edits.Pop());
edits.Push("rename sheet");
Console.WriteLine(edits.Peek());
Console.WriteLine(edits.Pop());
Console.WriteLine($"{edits.Pop()} / {edits.Count}");
```

Check your prediction against the solution, which runs the same statements.

In [ ]:
// Solution
using System;
using System.Collections.Generic;

IStack<string> edits = new ListStack<string>();
edits.Push("insert row");                     // [insert row]
edits.Push("format cell");                    // [insert row, format cell]
edits.Push("delete column");                  // [insert row, format cell, delete column]
Console.WriteLine(edits.Pop());               // removes delete column
edits.Push("rename sheet");                   // [insert row, format cell, rename sheet]
Console.WriteLine(edits.Peek());              // shows rename sheet, keeps it
Console.WriteLine(edits.Pop());               // removes rename sheet
Console.WriteLine($"{edits.Pop()} / {edits.Count}");   // removes format cell; one entry left

interface IStack<T> { int Count { get; } void Push(T item); T Pop(); T Peek(); }

class ListStack<T> : IStack<T>
{
    private readonly List<T> items = new();
    public int Count => items.Count;
    public void Push(T item) => items.Add(item);
    public T Pop()
    {
        if (items.Count == 0) throw new InvalidOperationException("Stack is empty.");
        T value = items[^1];
        items.RemoveAt(items.Count - 1);
        return value;
    }
    public T Peek() => items.Count == 0 ? throw new InvalidOperationException("Stack is empty.") : items[^1];
}

delete column
rename sheet
rename sheet
format cell / 1


The lines are `delete column`, `rename sheet`, `rename sheet`, and `format cell / 1`. The second and third lines print the same text because `Peek` reports without removing, so the following `Pop` returns the same entry. If you predicted `insert row` for the last line, you counted `Peek` as a removal; that is the most common tracing error with stacks and the reason the contract states the rule explicitly.

```{index} Selecting a Data Structure
```

### Selecting a Data Structure

Choosing a data structure is a design decision, and the ADT view gives it a procedure. First, analyze the problem to find the operations that must be supported and how often each is called. Second, quantify the resource limits: how many records, how much memory, how fast each operation must be, and whether the data arrives all at once or trickles in. Third, choose the data structure that meets those requirements best, knowing that every structure makes some operations cheap by making others expensive. A structure that is ideal for one workload is often wrong for another, so the question is never "which structure is best" but "best for these operations under these limits."

| Business task | Operations that dominate | Resource question | Reasonable first choice |
| --- | --- | --- | --- |
| Undo in a document editor | Add the newest action; remove the newest action | Depth is small; every keystroke must feel instant | Stack |
| Customer support tickets | Add at the back; serve from the front | Thousands per day; fairness matters | Queue |
| Product lookup by code | Find by key; occasional add and remove | Hundreds of thousands of codes; sub-millisecond lookups | Dictionary over a hash table |
| Monthly report sorted by region | Read all records in order once | Data is static after loading | Sorted array or list |

Three questions sharpen the first step. Are all records inserted at the start, or are inserts mixed with lookups and deletions? Can records be deleted at all? Are records processed in some fixed order, or is random access by key or position required? The answers often eliminate most candidates before any cost analysis begins. The remaining comparison is the subject of the next section.

Classify each of the following as an ADT or a data structure, then open the answer: list, array, stack, hash table, dictionary, linked nodes.

```{dropdown} Answer
List, stack, and dictionary are ADTs: each names operations and their results without committing to storage. Array, hash table, and linked nodes are data structures: each is a way of laying out memory that can implement one or more ADTs. A `List<T>` in the C# library is a class that pairs the list ADT with an array-backed data structure, so the library name hides both halves in one type.
```

```{index} Practice
```

## Practice



1. Write a contract for a queue ADT using operation names and meanings.
2. Name one invariant a queue implementation must preserve.
3. Explain why a method parameter typed as `IStack<string>` is more flexible than one typed as `ListStack<string>`.


```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. A teammate says, "Our `Inventory` class is a data structure, so it does not need a written contract." What is missing from that reasoning?

```{dropdown} Answer
Every data structure implements some ADT, whether or not anyone wrote it down. Without a contract, callers cannot know which operations are promised, what happens on an empty or missing item, or which invariants they may rely on. The class may be correct, but nobody can tell.
```

2. Two classes both implement `IStack<T>`. One returns `default` from `Pop` on an empty stack; the other throws. Do they satisfy the same contract?

```{dropdown} Answer
No. They share the interface, which is the syntactic part of the contract, but they differ in the empty-stack policy, which is part of the behavior. Client code written for one can silently misbehave with the other. The written contract must state the policy.
```

3. Which of the three rule kinds does this sentence express: "After `Enqueue(x)` on a queue, `Count` is one larger than before"?

```{dropdown} Answer
A postcondition. It describes the state after a specific operation returns, and the implementation is responsible for making it true.
```

4. Why is `static decimal Total(IEnumerable<Invoice> invoices)` more reusable than `static decimal Total(List<Invoice> invoices)`?

```{dropdown} Answer
The first depends only on the ability to visit the invoices in order, so it accepts arrays, lists, sets, query results, and custom collections. The second accepts one data structure. Depending on the smallest contract that the method actually needs is the same principle as depending on `IStack<T>` rather than `ListStack<T>`.
```

```{index} Summary
```

## Summary



An ADT names behavior. An interface can express that behavior in C#. Implementations provide storage and algorithms behind the contract. This separation lets programs change representation without changing every caller.


```{rubric} Footnotes
```


[^1301_adt_contracts-1]: Some C# library types use classes directly instead of exposing a small custom interface. The ADT idea still helps you reason about what behavior the code depends on.


[^development-reading]: Further reading: [OpenDSA CS3, Chapter Introduction: Lists](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/ListIntro.html). The worked business scenarios and guided reasoning here are original teaching examples.
